# Tomato Leaf Segmentation

1. Hãy Upload toàn bộ thư mục `Tomato Leaf Segmentation` lên Google Drive của bạn.
2. Đảm bảo cấu trúc thư mục dữ liệu và script giữ nguyên như trong repo.
3. Chạy các cell bên dưới để cài đặt môi trường và chuẩn bị dữ liệu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Tomato Leaf Segmentation

In [ ]:
# 3. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt
!pip install "numpy<2"  # Bắt buộc để tránh lỗi crash với PyTorch hiện tại

## Bắt đầu Huấn Luyện (Training)
Ở file `src/config.py`, số `EPOCHS` mặc định đã được set là 50. Bạn có thể mở file đó ra để chỉnh sửa nếu muốn train ít/nhiều hơn.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import sys
import os

# Thêm đường dẫn src
if '.' not in sys.path:
    sys.path.append('.')

from src import config
from src.data.transforms import get_train_transforms
from src.data.dataset import TomatoLeafDataset
from torch.utils.data import DataLoader

batch_size_view = min(4, config.BATCH_SIZE)

try:
    train_transform = get_train_transforms()
    train_dataset = TomatoLeafDataset(
        str(config.TRAIN_IMAGES_DIR),
        str(config.TRAIN_MASKS_DIR),
        transform=train_transform,
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size_view, shuffle=True)

    images, masks = next(iter(train_loader))

    mean = np.array(config.IMAGENET_MEAN)
    std = np.array(config.IMAGENET_STD)

    fig, axes = plt.subplots(2, batch_size_view, figsize=(16, 8))
    for i in range(batch_size_view):
        img_chw = images[i].permute(1, 2, 0).numpy()
        img_rgb = std * img_chw + mean
        img_rgb = np.clip(img_rgb, 0, 1)
        mask_np = masks[i].squeeze().numpy()

        axes[0, i].imshow(img_rgb)
        axes[0, i].set_title(f"Image (denorm) {i}")
        axes[0, i].axis('off')

        axes[1, i].imshow(mask_np, cmap='gray')
        axes[1, i].set_title(f"Mask {i}")
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Không thể hiển thị do lỗi dataset hoặc đường dẫn chưa tồn tại:", e)

In [ ]:
# 4.1 Huấn luyện U-Net
# !python scripts/train_unet.py

In [ ]:
# Dọn dẹp GPU Cache trước khi chuyển qua train mô hình mới để tránh lỗi Out Of Memory
import torch
import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# 4.2 Huấn luyện SegNet
!python scripts/train_segnet.py

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# 4.3 Huấn luyện U-Net++
# !python scripts/train_unetplusplus.py

## Đánh Giá & Xuất Biểu Đồ (Evaluation)

In [ ]:
# 5. Tính toán các chỉ số và xuất biểu đồ so sánh
# !python src/eval/evaluate_models.py
# !python src/eval/visualize.py

In [ ]:
# 6. Hiển thị kết quả trực tiếp ngay trên màn hình Colab
# from IPython.display import Image, display
# print("Biểu đồ so sánh Metrics:")
# display(Image(filename='docs/visualizations/metrics_comparison_bar.png'))
# print("\nKết quả phân vùng ảnh thực tế:")
# display(Image(filename='docs/visualizations/prediction_samples.png'))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import os

# ==========================================
# 1. BIỂU ĐỒ LEARNING CURVES (LOSS & IOU)
# ==========================================
def plot_learning_curves(csv_log_path, model_name="U-Net"):
    """
    Vẽ đường cong hội tụ từ file CSV log.
    Giả định file CSV có các cột: epoch, train_loss, val_loss, train_iou, val_iou
    """
    if not os.path.exists(csv_log_path):
        print(f"⚠️ Không tìm thấy file log: {csv_log_path}")
        print("💡 Gợi ý: Yêu cầu bước Train code của bạn lưu log ra CSV (VD: DataFrame.to_csv)")
        return
        
    df = pd.read_csv(csv_log_path)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Đồ thị Loss
    ax1.plot(df['epoch'], df['train_loss'], label='Train Loss', color='#1f77b4', linewidth=2)
    ax1.plot(df['epoch'], df['val_loss'], label='Validation Loss', color='#ff7f0e', linewidth=2, linestyle='--')
    ax1.set_title(f'Learning Curve: Loss ({model_name})', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epochs', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.legend(fontsize=11)
    ax1.grid(True, linestyle=':', alpha=0.6)
    
    # Đồ thị IoU
    if 'train_iou' in df.columns and 'val_iou' in df.columns:
        ax2.plot(df['epoch'], df['train_iou'], label='Train IoU', color='#2ca02c', linewidth=2)
        ax2.plot(df['epoch'], df['val_iou'], label='Validation IoU', color='#d62728', linewidth=2, linestyle='--')
        ax2.set_title(f'Learning Curve: IoU ({model_name})', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Epochs', fontsize=12)
        ax2.set_ylabel('IoU Score', fontsize=12)
        ax2.legend(fontsize=11)
        ax2.grid(True, linestyle=':', alpha=0.6)
        
    plt.tight_layout()
    plt.savefig(f'learning_curves_{model_name.lower()}.png', dpi=300)
    plt.show()

# ==========================================
# 2. BIỂU ĐỒ BOXPLOT (PHÂN BỐ TÍNH ỔN ĐỊNH)
# ==========================================
def plot_iou_distribution_boxplot(iou_dict):
    """
    Vẽ Boxplot so sánh độ phân tán IoU của các model trên tập Test.
    iou_dict: Dict chứa mảng IoU của từng model, vd: {'U-Net': [0.8, 0.85...], 'SegNet': [...]}
    """
    plt.figure(figsize=(10, 6))
    
    # Đưa dict vào DataFrame để Seaborn tự vẽ
    df_box = pd.DataFrame(iou_dict)
    
    sns.boxplot(data=df_box, palette="Set2", linewidth=1.5)
    sns.swarmplot(data=df_box, color=".25", size=4, alpha=0.6) # Thêm cá chấm nhỏ đại diện từng ảnh
    
    plt.title('Tính ổn định (Distribution) của từng Mô hình trên tập Test', fontsize=15, fontweight='bold')
    plt.ylabel('Test IoU Score', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0.0, 1.05)
    
    plt.tight_layout()
    plt.savefig('iou_distribution_boxplot.png', dpi=300)
    plt.show()

# --- MOCK DATA ĐỂ BẠN CHẠY THỬ XEM TRƯỚC BIỂU ĐỒ ---
print("Đang vẽ thử biểu đồ Boxplot mô phỏng...")
mock_iou_data = {
    'U-Net': np.random.normal(0.85, 0.05, 100),
    'SegNet': np.random.normal(0.82, 0.08, 100),
    'U-Net++': np.random.normal(0.91, 0.03, 100) # Phân tán hẹp nhất, ổn định nhất
}
plot_iou_distribution_boxplot(mock_iou_data)